In [ ]:
# %% Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Lasso, LassoCV, lasso_path
from sklearn.metrics import r2_score

import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX

# %% Dados: AirPassengers (mesmo conjunto do R)
# baixa diretamente do repositório de datasets do R via statsmodels
ap = sm.datasets.get_rdataset("AirPassengers", "datasets", cache=True).data
# a coluna no R chama-se "value" (às vezes "x"); padronizamos como 'x'
tbl = ap.rename(columns={ap.columns[-1]: "x"}).copy()

# cria coluna de datas mensais de 1949-01 a 1960-12
tbl["Date"] = pd.date_range(start="1949-01-01", end="1960-12-01", freq="MS")

# %% Gráficos básicos (opcional)
plt.figure()
plt.plot(tbl["Date"], tbl["x"])
plt.title("AirPassengers")
plt.xlabel("Date"); plt.ylabel("Passengers")
plt.tight_layout()

# %% Transformações
tbl["t"] = np.arange(1, len(tbl) + 1)
tbl["lnx"] = np.log(tbl["x"])

plt.figure()
plt.plot(tbl["Date"], tbl["lnx"])
plt.title("log(AirPassengers)")
plt.xlabel("Date"); plt.ylabel("ln(x)")
plt.tight_layout()

# %% Criando lags de lnx (1 a 30)
for i in range(1, 31):
    tbl[f"lnx_{i}"] = tbl["lnx"].shift(i)

# remove linhas com NA geradas pelos lags
tbl2 = tbl.dropna().reset_index(drop=True)

# y = lnx (coluna alvo)
y = tbl2["lnx"].to_numpy()

# X = t e lags 1..30 (equivalente a c(3,5:34) no R)
lag_cols = [f"lnx_{i}" for i in range(1, 31)]
X = tbl2[["t"] + lag_cols].to_numpy()

# %% Lasso com validação cruzada (7-fold), sem padronizar (como standardize=FALSE)
# Observação: scikit-learn normaliza por padrão somente se você usar um scaler antes.
# Aqui não padronizamos, para espelhar o script R.
alphas = np.logspace(-4, 4, 100)
lasso_cv = LassoCV(alphas=alphas, cv=7, fit_intercept=False, random_state=0)
lasso_cv.fit(X, y)

# curva de CV (MSE por alpha)
plt.figure()
plt.semilogx(lasso_cv.alphas_, lasso_cv.mse_path_.mean(axis=1))
plt.xlabel("alpha (lambda)"); plt.ylabel("CV MSE (mean)")
plt.title("Lasso CV")
plt.tight_layout()

print("Alphas testados (lambda):")
print(lasso_cv.alphas_)
print("\nMelhor alpha (lambda.min):", lasso_cv.alpha_)

# %% Ajuste final no melhor lambda e métricas
model_cv = Lasso(alpha=lasso_cv.alpha_, fit_intercept=False)
model_cv.fit(X, y)

y_hat = model_cv.predict(X)
SSR = np.dot((y - y_hat), (y - y_hat))
R2 = r2_score(y, y_hat)

print(f"\n=== Modelo Lasso (lambda.min) ===")
print("SSR:", float(SSR))
print("R^2:", float(R2))
print("\nCoeficientes (t, lnx_1..lnx_30):")
for name, coef in zip(["t"] + lag_cols, model_cv.coef_):
    print(f"{name:>6}: {coef: .6f}")

# %% Caminho de coeficientes vs lambda (equivalente ao plot(res, xvar='lambda'))
# Usando lasso_path para obter o caminho completo
_, coefs, _ = lasso_path(X, y, alphas=alphas, fit_intercept=False)

plt.figure()
for i in range(coefs.shape[0]):
    plt.semilogx(alphas, coefs[i, :])
plt.gca().invert_xaxis()   # para decrescer como no glmnet
plt.xlabel("alpha (lambda)")
plt.ylabel("coeficientes")
plt.title("Caminho dos coeficientes - Lasso")
plt.tight_layout()

# %% (Opcional) Ajustar com um lambda específico (ex.: 1.261857e-02 do script R)
lambda_fixed = 1.261857e-02
model_fixed = Lasso(alpha=lambda_fixed, fit_intercept=False)
model_fixed.fit(X, y)

y_hat_f = model_fixed.predict(X)
SSR_f = np.dot((y - y_hat_f), (y - y_hat_f))
R2_f = r2_score(y, y_hat_f)

print(f"\n=== Modelo Lasso (lambda fixo = {lambda_fixed}) ===")
print("SSR:", float(SSR_f))
print("R^2:", float(R2_f))

# %% Regressão linear: lnx ~ t + lnx_12 - 1 (sem intercepto)
tbl_ols = tbl.dropna().copy()
tbl_ols = tbl_ols.assign(lnx_12=tbl_ols["lnx"].shift(12)).dropna()
Y_ols = tbl_ols["lnx"].to_numpy()
X_ols = tbl_ols[["t", "lnx_12"]].to_numpy()
# sem intercepto: não adicionamos constante
ols_res = sm.OLS(Y_ols, X_ols).fit()
print("\n=== OLS: lnx ~ t + lnx_12 - 1 ===")
print(ols_res.summary())

# (opcional) gráficos de diagnóstico simples
fig = plt.figure()
plt.plot(ols_res.resid)
plt.title("Resíduos OLS")
plt.tight_layout()

# %% ARIMA sazonal (0,0,0) com sazonal (1,0,0)[12], exógena t, sem média (trend='n')
# Em statsmodels, usamos SARIMAX com seasonal_order e exog
tbl_arima = tbl.copy()
tbl_arima["t"] = np.arange(1, len(tbl_arima) + 1)
# alinhar exógena com série lnx
endog = tbl_arima["lnx"].dropna()
exog = tbl_arima["t"].iloc[endog.index]

sarimax_mod = SARIMAX(endog,
                      exog=exog,
                      order=(0, 0, 0),
                      seasonal_order=(1, 0, 0, 12),
                      trend='n',
                      enforce_stationarity=False,
                      enforce_invertibility=False)
sarimax_res = sarimax_mod.fit(disp=False)
print("\n=== SARIMAX (0,0,0) x (1,0,0)[12] com exógena t, sem média ===")
print(sarimax_res.summary())

# (opcional) previsões in-sample
fitted = sarimax_res.fittedvalues
plt.figure()
plt.plot(tbl_arima["Date"].iloc[endog.index], endog, label="lnx")
plt.plot(tbl_arima["Date"].iloc[endog.index], fitted, label="ajustado")
plt.legend(); plt.title("SARIMAX: ajuste in-sample")
plt.tight_layout()

plt.show()
